In [27]:
import gzip
import numpy as np
import matplotlib.pyplot as plt
import torch

def get_corr(vector_list): # vector_list: [element_id, worker_id, sample_id_(ta_id, batch_idx)]
    return [np.array([np.correlate(e[0], e[1])[0], 0]) for e in vector_list]

train_attempt_count, worker_count, round_count, epoch_count, batch_count = 4, 2, 2, 30, 17

path_to_files = [f"exp_data/gradients_resnet/gradients_resnet_t{i}/" for i in range(0, 4)]

filename = f"_round_{0}_epoch_{0}_batch_{0}_gradients.pt.gz"
with gzip.open(path_to_files[0] + f"worker_{0}" + filename, "rb") as f:
    temp1 = torch.load(f)
    layer_counts = len(temp1)
    per_layer_ele_count = [len(t) for t in temp1]

temp=(path_to_files[0] + f"_grad_namings.txt")
with open(temp, "rb") as f:
    layer_names = f.read().decode("utf-8").replace("\r",'').split("\n")[:-1]

In [28]:
# {layer_id}, time_(round, epoch), element_id, corr_(corr, MI, dist, cos)
result = {k:[] for k in layer_names}
time_steps = np.array(np.meshgrid(range(round_count), range(epoch_count))).T.reshape(-1, 2)
for curr_round, current_epoch in time_steps:
    print(f"Round {curr_round}, Epoch {current_epoch}")

    # {layer_id}, sample_id_(ta_id, batch_idx), worker_id, element_id_(diff size per layer)
    sample_dict = {k:[] for k in layer_names}
    sample_steps = np.array(np.meshgrid(range(train_attempt_count), range(batch_count))).T.reshape(-1, 2)
    for train_attempt, batch_idx in sample_steps:
        filename = f"_round_{curr_round}_epoch_{current_epoch}_batch_{batch_idx}_gradients.pt.gz"
        with gzip.open(path_to_files[train_attempt] + f"worker_{0}" + filename, "rb") as f:
            grads_w_1 = torch.load(f)
        with gzip.open(path_to_files[train_attempt] + f"worker_{1}" + filename, "rb") as f:
            grads_w_2 = torch.load(f)
        f = lambda x,i: x[i].numpy().flatten()
        sample_dict= {k:sample_dict[k]+[(f(grads_w_1,i),f(grads_w_2,i))]
                      for i,k in enumerate(layer_names)}
    # Transpose to have shape: {layer_id} [element_id, worker_id, sample_id_(ta_id, batch_idx)]
    sample_dict = {k:np.array(sample_dict[k]).transpose(2,1,0) for k in sample_dict.keys()}

    result = {k:result[k] + [get_corr(sample_dict[k])] for k in sample_dict.keys()}
result = {k: np.array(result[k]) for k in layer_names}

Round 0, Epoch 0
